In [2]:
import pandas as pd
import sklearn_crfsuite
from seqeval.metrics import classification_report
import spacy
from spacy.tokens import Doc
from spacy import displacy

In [3]:
def load_conll_data(filepath):
    sentences = []
    labels = []
    
    sentence = []
    label = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip() == "":
                if sentence:
                    sentences.append(sentence)
                    labels.append(label)
                    sentence = []
                    label = []
            else:
                parts = line.strip().split()
                word = parts[0]
                ner_tag = parts[-1]
                sentence.append(word)
                label.append(ner_tag)
    
    return sentences, labels

X_train, y_train = load_conll_data("train.txt")
X_valid, y_valid = load_conll_data("valid.txt")
X_test, y_test = load_conll_data("test.txt")

In [4]:
print("Training Sentences:", len(X_train))
print("Example Sentence:", X_train[0])
print("Example Labels:", y_train[0])

Training Sentences: 14987
Example Sentence: ['-DOCSTART-']
Example Labels: ['O']


In [5]:
def rule_based(sentence):
    predictions = []
    for word in sentence:
        if word.istitle():
            predictions.append("B-PER")
        else:
            predictions.append("O")
    return predictions

In [6]:
sample_sentence = X_test[5]
rule_pred = rule_based(sample_sentence)

for w, t in zip(sample_sentence, rule_pred):
    print(w, "->", t)

But -> B-PER
China -> B-PER
saw -> O
their -> O
luck -> O
desert -> O
them -> O
in -> O
the -> O
second -> O
match -> O
of -> O
the -> O
group -> O
, -> O
crashing -> O
to -> O
a -> O
surprise -> O
2-0 -> O
defeat -> O
to -> O
newcomers -> O
Uzbekistan -> B-PER
. -> O


In [7]:
def word2features(sentence, i):
    word = sentence[i]
    
    features = {
        'word.lower()': word.lower(),
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    
    if i > 0:
        prev_word = sentence[i-1]
        features.update({
            '-1:word.lower()': prev_word.lower(),
            '-1:word.istitle()': prev_word.istitle(),
        })
    else:
        features['BOS'] = True
        
    if i < len(sentence)-1:
        next_word = sentence[i+1]
        features.update({
            '+1:word.lower()': next_word.lower(),
            '+1:word.istitle()': next_word.istitle(),
        })
    else:
        features['EOS'] = True
        
    return features


def prepare_features(sentences):
    return [[word2features(sentence, i) for i in range(len(sentence))]
            for sentence in sentences]

In [8]:
X_train_features = prepare_features(X_train)
X_test_features = prepare_features(X_test)

In [9]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100
)

crf.fit(X_train_features, y_train)

CRF(algorithm='lbfgs', max_iterations=100)

In [10]:
y_pred = crf.predict(X_test_features)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         LOC       0.84      0.75      0.79      1668
        MISC       0.77      0.66      0.71       702
         ORG       0.74      0.59      0.66      1661
         PER       0.79      0.82      0.81      1617

   micro avg       0.79      0.71      0.75      5648
   macro avg       0.79      0.71      0.74      5648
weighted avg       0.79      0.71      0.75      5648



In [11]:
import spacy
from spacy.tokens import Doc, Span
from spacy import displacy

nlp = spacy.blank("en")

sentence = X_test[10]
prediction = y_pred[10]

doc = Doc(nlp.vocab, words=sentence)

entities = []
start = None
label = None

for i, tag in enumerate(prediction):
    if tag.startswith("B-"):
        if start is not None:
            entities.append((start, i, label))
        start = i
        label = tag[2:]
    elif tag.startswith("I-"):
        continue
    else:
        if start is not None:
            entities.append((start, i, label))
            start = None

if start is not None:
    entities.append((start, len(prediction), label))

doc.ents = [Span(doc, start, end, label=label) for start, end, label in entities]

displacy.render(doc, style="ent", jupyter=True)